# Mercy Fantasy

## Set Up

In [1]:
import yahoo_fantasy_api as yfa
import pandas as pd
from yahoo_oauth import OAuth2
from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, String, Float, Boolean, Date, BigInteger
from datetime import datetime

oauth = OAuth2(None, None, from_file="oauth2.json")
if not oauth.token_is_valid():
    oauth.refresh_access_token()
gm = yfa.Game(oauth, "nfl")

[2025-11-11 19:23:03,427 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2025-11-11 19:23:03,432 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 432853.52890872955
[2025-11-11 19:23:03,433 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN HAS EXPIRED
[2025-11-11 19:23:03,434 DEBUG] [yahoo_oauth.oauth.refresh_access_token] REFRESHING TOKEN
[2025-11-11 19:23:03,777 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 0.34010887145996094
[2025-11-11 19:23:03,779 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


In [2]:
idMap = {2023: '423.l.927065', 2024: '449.l.978032', 2025: '461.l.1390291'}

In [3]:
# replace with your actual database info
user = "mercyfantasy"
password = "Matthew23"
host = "localhost"       # or remote IP / domain
port = "5432"
database = "mercyfantasy"

# create a database connection
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{database}")

## Historical Data

In [3]:
for year in (2023, 2024, 2025):
    league_ids = gm.league_ids(year)

    for lid in league_ids:
        lg = gm.to_league(lid)
        settings = lg.settings()  # includes league name + details
        print(f"League ID: {lid} | Name: {settings['name']} | Season: {settings['season']}")

League ID: 423.l.120539 | Name: New Years Weigh In | Season: 2023
League ID: 423.l.927065 | Name: DEM HICKSVILLE BOYS | Season: 2023
League ID: 449.l.91073 | Name: nick's Unbelievable League | Season: 2024
League ID: 449.l.978032 | Name: MERCY FANTASY | Season: 2024
League ID: 461.l.1135347 | Name: Mercy Fantasy | Season: 2025
League ID: 461.l.1388330 | Name: chris's Notable League | Season: 2025
League ID: 461.l.1390291 | Name: Montauk Monster | Season: 2025


In [ ]:
# Populate this array with each team's data for each matchup
rows = []
managers = []

# Iterate over each league and each week to get all matchups
for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    settings = lg.settings()
    end_week = int(settings["end_week"])
    
    for week in range(1, end_week + 1):
        matchup = lg.matchups(week)["fantasy_content"]["league"][1]['scoreboard']
        for key, value in matchup['0']['matchups'].items():
            if key != 'count':
                match = value['matchup']
                if match['is_matchup_recap_available'] != 0:
                    week = match['week']
                    week_start = match['week_start']
                    week_end = match['week_end']
                    isPlayoff = match['is_playoffs']
                    teams = match['0']['teams']
                    
                    team1 = teams['0']['team'][0][2]['name']
                    team2 = teams['1']['team'][0][2]['name']

                    points1 = float(teams['0']['team'][1]['team_points']['total'])
                    points2 = float(teams['1']['team'][1]['team_points']['total'])

                    # Chat
                    recap_url = match['matchup_recap_url']
                    week = int(match['week'])
                    winner_key = match['winner_team_key']

                    # Each matchup has exactly two teams: '0' and '1'
                    for side in ['0', '1']:
                        team = teams[side]['team']
                        team_data = team[0]   # long nested metadata list
                        scoring = team[1]     # dict with points and projections

                        # --- Extract basic info ---
                        team_key = team_data[0]['team_key']
                        team_id = team_data[1]['team_id']
                        team_name = team_data[2]['name']
                        points = float(scoring['team_points']['total'])
                        proj_points = float(scoring['team_projected_points']['total'])

                        # --- Winner flag ---
                        winner = team_key == winner_key

                        # --- Extract manager info ---
                        manager_info = None
                        for item in team_data:
                            if isinstance(item, dict) and 'managers' in item:
                                manager_info = item['managers'][0]['manager']
                                break

                        manager_id = manager_info.get('manager_id') if manager_info else None
                        nickname = manager_info.get('nickname') if manager_info else None
                        guid = manager_info.get('guid') if manager_info else None

                        if week == 1:
                            num_moves, num_trades = None, None
                            for item in team_data:
                                if isinstance(item, dict):
                                    if 'number_of_moves' in item:
                                        num_moves = int(item['number_of_moves'])
                                    elif 'number_of_trades' in item:
                                        try:
                                            num_trades = int(item['number_of_trades'])
                                        except ValueError:
                                            num_trades = None
                            managers.append({
                                "manager_guid": guid,
                                "season": year,
                                "league_id": league_id,
                                "manager_id": manager_id,
                                "manager_nickname": nickname,
                                "team_id": team_id,
                                "team_key": team_key,
                                "number_of_moves": num_moves,
                                "number_of_trades": num_trades
                            })

                        # --- Add a flattened record ---
                        rows.append({
                            "week": week,
                            "week_start": week_start,
                            "week_end": week_end,
                            "is_playoff": isPlayoff,
                            "matchup_id": key,
                            "team_id": team_id,
                            "team_name": team_name,
                            "team_key": team_key,
                            "manager_id": manager_id,
                            "manager_nickname": nickname,
                            "manager_guid": guid,
                            "points": points,
                            "projected_points": proj_points,
                            "winner": winner,
                            "recap_url": recap_url
                        })

matchups_df = pd.DataFrame(rows)
managers_df = pd.DataFrame(managers)

In [147]:
matchups_df.tail(5)

,week,week_start,week_end,is_playoff,matchup_id,team_id,team_name,team_key,manager_id,manager_nickname,manager_guid,points,projected_points,winner,recap_url
359,4,2025-09-23,2025-09-29,0,2,8,Smart Play Matthew,461.l.1390291.t.8,8,Matthew,GOBJY4A3RDGQYERM2ETTMRC7YY,128.48,130.97,False,https://football.fantasysports.yahoo.com/f1/13...
360,4,2025-09-23,2025-09-29,0,3,5,Amon Meth,461.l.1390291.t.5,5,Billy,Z7QJUKFA7ZPHDWIVNLXNVZZBVU,164.76,144.68,True,https://football.fantasysports.yahoo.com/f1/13...
361,4,2025-09-23,2025-09-29,0,3,9,Liam’s Ice,461.l.1390291.t.9,9,Liam,BYOKM33VSC7PSCEB74CKX2E6FQ,149.10,139.95,False,https://football.fantasysports.yahoo.com/f1/13...
362,4,2025-09-23,2025-09-29,0,4,6,Hail Mary john,461.l.1390291.t.6,6,john,BB5X53X74V3GT4BZO4W4ZE3S2I,112.62,114.65,False,https://football.fantasysports.yahoo.com/f1/13...
363,4,2025-09-23,2025-09-29,0,4,7,Don Toes Legitimate Football A,461.l.1390291.t.7,7,Joseph,ZFECT2SSXQHCUOGITGLWEYRHJE,131.50,134.51,True,https://football.fantasysports.yahoo.com/f1/13...


In [148]:
managers_df.head(5)

,manager_guid,season,league_id,manager_id,manager_nickname,team_id,number_of_moves,number_of_trades
0,RLDAQUDDIH63SCRVGDUH7XHVVE,2023,423.l.927065,1,chris,1,65,10
1,GOBJY4A3RDGQYERM2ETTMRC7YY,2023,423.l.927065,5,Matthew,5,15,0
2,HL2OCHLAPFDTIFOI3ZENZETP5I,2023,423.l.927065,2,Skyler,2,33,3
3,2LZKPL56AY3GJ5QCXA6HWZOMGA,2023,423.l.927065,8,nick,8,21,5
4,Z7QJUKFA7ZPHDWIVNLXNVZZBVU,2023,423.l.927065,3,Billy,3,12,5


In [ ]:
all_players = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    settings = lg.settings()
    end_week = int(settings["end_week"])
    teams = lg.teams()
    for week in range(1, end_week + 1):
        for team_name, team in teams.items():
            roster = lg.to_team(team_name).roster(week=week)
            playerIds = [player["player_id"] for player in roster]
            stats = lg.player_stats(playerIds, req_type='week', week=week)            
            for player, stat in zip(roster, stats):
                all_players.append({
                    "season": year,
                    "week": week,
                    "team_key": team_name,
                    "team_name": team['name'],
                    "manager_guid": team['managers'][0]['manager']['guid'],
                    "manager_name": team['managers'][0]['manager']['nickname'],
                    "player_id": player["player_id"],
                    "player_name": player["name"],
                    "position": player["selected_position"],
                    "points": stat['total_points'],
                })

roster = pd.DataFrame(all_players)

In [193]:
managers_df.to_sql('managers', engine, if_exists='replace', index=False, dtype={
    "manager_guid": String(50),      # unique Yahoo GUID (e.g., 'GOBJY4A3RDGQYERM2ETTMRC7YY')
    "season": Integer(),             # year (e.g., 2025)
    "league_id": String(50),          # numeric league ID
    "manager_id": Integer(),         # Yahoo manager ID
    "manager_nickname": String(50),  # manager's nickname ('Matthew')
    "team_id": Integer(),            # numeric team ID
    "number_of_moves": Integer(),    # total moves made
    "number_of_trades": Integer()    # total trades made
    })

matchups_df.to_sql('matchups', engine, if_exists='replace', index=False, dtype={
    "week": Integer(),              # week number (1–18)
    "week_start": Date(),           # start date of the week
    "week_end": Date(),             # end date of the week
    "is_playoff": Integer(),        # True if this week is in playoffs
    "matchup_id": Integer(),        # Yahoo matchup ID (numeric)
    "team_id": Integer(),           # numeric team ID
    "team_name": String(100),       # e.g., "Hail Mary john"
    "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
    "manager_id": Integer(),        # numeric manager ID (if available)
    "manager_nickname": String(50), # e.g., "Matthew"
    "manager_guid": String(50),     # e.g., "GOBJY4A3RDGQYERM2ETTMRC7YY"
    "points": Float(),              # actual fantasy points scored
    "projected_points": Float(),    # projected points for the matchup
    "winner": String(10),           # 'win', 'loss', 'tie', or None
    "recap_url": String(255),       # URL to Yahoo recap (can be long)
})

roster.to_sql('roster', engine, if_exists='replace', index=False, dtype={
    "season": Integer(),          # year like 2025
    "week": Integer(),            # week number (1–18)
    "team_key": String(50),       # Yahoo team key (short ID)
    "team_name": String(100),     # team name ("Hail Mary john")
    "manager_guid": String(50),   # unique Yahoo GUID ("GOBJY4A3RDGQYERM2ETTMRC7YY")
    "manager_name": String(50),   # nickname of manager ("Matthew")
    "player_id": Integer(),       # numeric Yahoo player ID (e.g., 31002)
    "player_name": String(100),   # player full name ("Lamar Jackson")
    "position": String(10),       # e.g., QB, RB, WR, TE, etc.
    "points": Float(),            # fantasy points (decimal)
})

510

### Transactions

In [162]:
parsed_trades = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    transactions = lg.transactions('trade', '1000')
    for trade in transactions:
        trade_id = trade['transaction_id']
        status = trade['status']
        timestamp = int(trade['timestamp'])
        players = trade['players']

        for k, player_entry in players.items():
            if k == 'count':
                continue

            # player info
            player_info = player_entry['player'][0]
            transaction_info = player_entry['player'][1]['transaction_data'][0]

            player_name = player_info[2]['name']['full']
            played_id = player_info[1]['player_id']
            position = player_info[4]['display_position']
            team_abbr = player_info[3]['editorial_team_abbr']

            from_team = transaction_info['source_team_name']
            from_team_key = transaction_info['source_team_key']
            to_team = transaction_info['destination_team_name']
            to_team_key = transaction_info['destination_team_key']

            parsed_trades.append({
                'season': year,
                'league_id': league_id,
                'trade_id': trade_id,
                'timestamp': timestamp,
                'status': status,
                'from_team': from_team,
                'from_team_key': from_team_key,
                'to_team': to_team,
                'to_team_key': to_team_key,
                'player': player_name,
                'player_id': played_id,
                'pos': position,
                'team_abbr': team_abbr
            })

trades = pd.DataFrame(parsed_trades)

In [163]:
trades.to_sql('trades', engine, if_exists='replace', index=False, dtype={
    "season": Integer(),            # year like 2025
    "league_id": String(50),        # league key (short ID)
    "trade_id": Integer(),         # trade or transaction ID
    "timestamp": BigInteger(),      # UNIX timestamp of transaction
    "status": String(50),           # e.g., 'successful'
    "from_team": String(100),       # name of team initiating transaction
    "from_team_key": String(100),   # key of team initiating transaction
    "to_team": String(100),         # name of team receiving player
    "to_team_key": String(100),     # key of team receiving player
    "player": String(100),          # player name
    "player_id": Integer(),        # player ID
    "pos": String(10),              # player position, e.g., 'DEF'
    "team_abbr": String(10)         # team abbreviation, e.g., 'CHI'
})

237

In [154]:
parsed_waivers = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    transactions = lg.transactions('add', '5000')
    for trade in transactions:
        trade_id = trade['transaction_id']
        status = trade['status']
        timestamp = int(trade['timestamp'])
        players = trade['players']

        for k, player_entry in players.items():
            if k == 'count':
                continue

            # player info
            player_info = player_entry['player'][0]
            transaction_info = player_entry['player'][1]['transaction_data'][0] if isinstance(player_entry['player'][1]['transaction_data'], list) else player_entry['player'][1]['transaction_data']

            player_name = player_info[2]['name']['full']
            played_id = player_info[1]['player_id']
            position = player_info[4]['display_position']
            team_abbr = player_info[3]['editorial_team_abbr']
            type = transaction_info['type']
            add = type == 'add'
            teamType = 'destination' if add else 'source'

            team = transaction_info[f'{teamType}_team_name'] 
            team_key = transaction_info[f'{teamType}_team_key']

            parsed_waivers.append({
                'season': year,
                'league_id': league_id,
                'trade_id': trade_id,
                'timestamp': timestamp,
                'status': status,
                'type': type,
                'team': team,
                'team_key': team_key,
                'player': player_name,
                'player_id': played_id,
                'pos': position,
                'team_abbr': team_abbr
            })

waivers = pd.DataFrame(parsed_waivers)

In [155]:
waivers.to_sql('waivers', engine, if_exists='replace', index=False, dtype={
    "season": Integer(),           # year like 2025
    "league_id": String(50),       # league key (short ID)
    "trade_id": String(50),        # trade or transaction ID
    "timestamp": BigInteger(),     # UNIX timestamp of transaction
    "status": String(50),          # e.g., 'successful'
    "type": String(50),            # transaction type, e.g., 'add/drop'
    "team": String(100),           # team name
    "team_key": String(100),       # team key
    "player": String(100),         # player name
    "player_id": String(50),       # player ID
    "pos": String(10),             # player position, e.g., 'DEF'
    "team_abbr": String(10)        # team abbreviation, e.g., 'CHI'
})

418

In [98]:
rows = []

for year, league_id in idMap.items():
    lg = gm.to_league(league_id)
    settings = lg.settings()
    end_week = int(settings["end_week"])
    teams = lg.teams()
    for team_name, team in teams.items(): 
        rows.append({
            'Season': year,
            'league_id': league_id,
            'team_key': team_name,
            'manager_id': team['managers'][0]['manager']['manager_id'],
            'nickname': team['managers'][0]['manager']['nickname'],
            'guid': team['managers'][0]['manager']['guid']
        })

teamsDf = pd.DataFrame(rows)

In [156]:
teamsDf.to_sql('teams', engine, if_exists='replace', index=False, dtype={
    "season": Integer(),         # year like 2025
    "league_id": String(50),     # Yahoo team key (short ID)
    "team_key": String(100),     # team key 
    "manager_id": Integer(),     # managerid
    "nickname": String(15),      # nickname
    "guid": String(50)           # unique Yahoo GUID ("GOBJY4A3RDGQYERM2ETTMRC7YY")
})

30

## Important Weekly Runs

In [4]:
currentYear = datetime.now().year

In [5]:
lg = gm.to_league(idMap[currentYear])
settings = lg.settings()
current_week = lg.current_week()

### Current Matchup

In [6]:
rows = []
matchup = lg.matchups(current_week)["fantasy_content"]["league"][1]['scoreboard']

for key, value in matchup['0']['matchups'].items():
    if key != 'count':
        match = value['matchup']
        week = match['week']
        week_start = match['week_start']
        week_end = match['week_end']
        isPlayoff = match['is_playoffs']
        teams = match['0']['teams']
        
        team1 = teams['0']['team'][0][2]['name']
        team2 = teams['1']['team'][0][2]['name']

        # Each matchup has exactly two teams: '0' and '1'
        for side in ['0', '1']:
            team = teams[side]['team']
            team_data = team[0]   # long nested metadata list
            scoring = team[1]     # dict with points and projections

            # --- Extract basic info ---
            team_key = team_data[0]['team_key']
            team_id = team_data[1]['team_id']
            team_name = team_data[2]['name']
            proj_points = float(scoring['team_projected_points']['total'])

            manager_info = None
            for item in team_data:
                if isinstance(item, dict) and 'managers' in item:
                    manager_info = item['managers'][0]['manager']
                    break

            manager_id = manager_info.get('manager_id') if manager_info else None
            nickname = manager_info.get('nickname') if manager_info else None
            guid = manager_info.get('guid') if manager_info else None

            # --- Add a flattened record ---
            rows.append({
                "week": week,
                "week_start": week_start,
                "week_end": week_end,
                "is_playoff": isPlayoff,
                "matchup_id": key,
                "team_id": team_id,
                "team_name": team_name,
                "team_key": team_key,
                "manager_id": manager_id,
                "manager_nickname": nickname,
                "manager_guid": guid,
                "projected_points": proj_points,
            })

currentMatchup = pd.DataFrame(rows)

In [7]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE currentmatchup RESTART IDENTITY CASCADE;"))

In [8]:
currentMatchup.to_sql('currentmatchup', engine, if_exists='append', index=False, dtype={
    "week": Integer(),              # week number (1–18)
    "week_start": Date(),           # start date of the week
    "week_end": Date(),             # end date of the week
    "is_playoff": Integer(),        # True if this week is in playoffs
    "matchup_id": Integer(),        # Yahoo matchup ID (numeric)
    "team_id": Integer(),           # numeric team ID
    "team_name": String(100),       # e.g., "Hail Mary john"
    "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
    "manager_id": Integer(),        # numeric manager ID (if available)
    "manager_nickname": String(50), # e.g., "Matthew"
    "manager_guid": String(50),     # e.g., "GOBJY4A3RDGQYERM2ETTMRC7YY"
    "projected_points": Float()    # projected points for the matchup
})

10

### Append Last Weeks Data

In [9]:
previousWeek = current_week - 1 if current_week != 17 else 17

In [10]:
rows = []
matchup = lg.matchups(previousWeek)["fantasy_content"]["league"][1]['scoreboard']

for key, value in matchup['0']['matchups'].items():
    if key != 'count':
        match = value['matchup']
        if match['is_matchup_recap_available'] != 0:
            week = match['week']
            week_start = match['week_start']
            week_end = match['week_end']
            isPlayoff = match['is_playoffs']
            teams = match['0']['teams']
            
            team1 = teams['0']['team'][0][2]['name']
            team2 = teams['1']['team'][0][2]['name']

            points1 = float(teams['0']['team'][1]['team_points']['total'])
            points2 = float(teams['1']['team'][1]['team_points']['total'])

            # Chat
            recap_url = match['matchup_recap_url']
            week = int(match['week'])
            winner_key = match['winner_team_key']

            # Each matchup has exactly two teams: '0' and '1'
            for side in ['0', '1']:
                team = teams[side]['team']
                team_data = team[0]   # long nested metadata list
                scoring = team[1]     # dict with points and projections

                # --- Extract basic info ---
                team_key = team_data[0]['team_key']
                team_id = team_data[1]['team_id']
                team_name = team_data[2]['name']
                points = float(scoring['team_points']['total'])
                proj_points = float(scoring['team_projected_points']['total'])

                # --- Winner flag ---
                winner = team_key == winner_key

                # --- Extract manager info ---
                manager_info = None
                for item in team_data:
                    if isinstance(item, dict) and 'managers' in item:
                        manager_info = item['managers'][0]['manager']
                        break

                manager_id = manager_info.get('manager_id') if manager_info else None
                nickname = manager_info.get('nickname') if manager_info else None
                guid = manager_info.get('guid') if manager_info else None

                if previousWeek == 1:
                    num_moves, num_trades = None, None
                    for item in team_data:
                        if isinstance(item, dict):
                            if 'number_of_moves' in item:
                                num_moves = int(item['number_of_moves'])
                            elif 'number_of_trades' in item:
                                try:
                                    num_trades = int(item['number_of_trades'])
                                except ValueError:
                                    num_trades = None
                    managers.append({
                        "manager_guid": guid,
                        "season": currentYear,
                        "league_id": league_id,
                        "manager_id": manager_id,
                        "manager_nickname": nickname,
                        "team_id": team_id,
                        "number_of_moves": num_moves,
                        "number_of_trades": num_trades
                    })

                # --- Add a flattened record ---
                rows.append({
                    "week": week,
                    "week_start": week_start,
                    "week_end": week_end,
                    "is_playoff": isPlayoff,
                    "matchup_id": key,
                    "team_id": team_id,
                    "team_name": team_name,
                    "team_key": team_key,
                    "manager_id": manager_id,
                    "manager_nickname": nickname,
                    "manager_guid": guid,
                    "points": points,
                    "projected_points": proj_points,
                    "winner": winner,
                    "recap_url": recap_url
                })

lastWeekMathcup = pd.DataFrame(rows)

In [11]:
lastWeekMathcup.to_sql('matchups', engine, if_exists='append', index=False, dtype={
    "week": Integer(),              # week number (1–18)
    "week_start": Date(),           # start date of the week
    "week_end": Date(),             # end date of the week
    "is_playoff": Integer(),        # True if this week is in playoffs
    "matchup_id": Integer(),        # Yahoo matchup ID (numeric)
    "team_id": Integer(),           # numeric team ID
    "team_name": String(100),       # e.g., "Hail Mary john"
    "team_key": String(50),         # Yahoo team key (e.g., "449.l.12345.t.1")
    "manager_id": Integer(),        # numeric manager ID (if available)
    "manager_nickname": String(50), # e.g., "Matthew"
    "manager_guid": String(50),     # e.g., "GOBJY4A3RDGQYERM2ETTMRC7YY"
    "points": Float(),              # actual fantasy points scored
    "projected_points": Float(),    # projected points for the matchup
    "winner": String(10),           # 'win', 'loss', 'tie', or None
    "recap_url": String(255),       # URL to Yahoo recap (can be long)
})

10

In [12]:
previousWeekPlayers = []
teams = lg.teams()

for team_name, team in teams.items():
    roster = lg.to_team(team_name).roster(week=previousWeek)
    playerIds = [player["player_id"] for player in roster]
    stats = lg.player_stats(playerIds, req_type='week', week=week)            
    for player, stat in zip(roster, stats):
        previousWeekPlayers.append({
            "season": currentYear,
            "week": week,
            "team_key": team_name,
            "team_name": team['name'],
            "manager_guid": team['managers'][0]['manager']['guid'],
            "manager_name": team['managers'][0]['manager']['nickname'],
            "player_id": player["player_id"],
            "player_name": player["name"],
            "position": player["selected_position"],
            "points": stat['total_points'],
        })

previousWeekRoster = pd.DataFrame(previousWeekPlayers)

In [13]:
previousWeekRoster.to_sql('roster', engine, if_exists='append', index=False, dtype={
    "season": Integer(),          # year like 2025
    "week": Integer(),            # week number (1–18)
    "team_key": String(50),       # Yahoo team key (short ID)
    "team_name": String(100),     # team name ("Hail Mary john")
    "manager_guid": String(50),   # unique Yahoo GUID ("GOBJY4A3RDGQYERM2ETTMRC7YY")
    "manager_name": String(50),   # nickname of manager ("Matthew")
    "player_id": Integer(),       # numeric Yahoo player ID (e.g., 31002)
    "player_name": String(100),   # player full name ("Lamar Jackson")
    "position": String(10),       # e.g., QB, RB, WR, TE, etc.
    "points": Float(),            # fantasy points (decimal)
})

173

## Reports

In [14]:
sql = text('''
Select S.*, 
HS.Season as Season_S, HS.Week as Week_S, HS.winningmanager as WinningManager_S, HS.winningpoints as WinningPoints_S,   
HS.losingmanager as LosingManager_S, HS.Losingpoints as LosingPoints_S,
HL.Season as Season_L, HL.Week as Week_L, HL.winningmanager as WinningManager_L, HL.winningpoints as WinningPoints_L,   
HL.losingmanager as LosingManager_L, HL.Losingpoints as LosingPoints_L
From (
Select M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0) as LifetimeWins, COALESCE(R.Losses, RFallBack.Wins, 0) as LifetimeLosses, COALESCE(Min(H.pointdiff), 0) as SmallestPointDifferential, COALESCE(Max(H.pointdiff), 0) as LargestPointDifferential
from vw_currentmatchup M
Left Join lifetimerecord R On M.projectedwinningmanager = R.winningmanager and M.projectedlosingmanager = R.losingmanager
Left Join lifetimerecord RFallBack On M.projectedwinningmanager = RFallBack.losingmanager and M.projectedlosingmanager = RFallBack.winningmanager 
Left Join matchuphistory H On (M.projectedwinningmanager = H.winningmanager or M.projectedwinningmanager = H.losingmanager) and (M.projectedlosingmanager = H.losingmanager or M.projectedlosingmanager = H.winningmanager)
Group By M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0), COALESCE(R.Losses, RFallBack.Wins, 0)
) S
Left Join matchuphistory HL On S.LargestPointDifferential = HL.pointdiff and (S.projectedwinningmanager = HL.winningmanager or S.projectedwinningmanager = HL.losingmanager) and (S.projectedlosingmanager = HL.losingmanager or s.projectedlosingmanager = HL.winningmanager)
Left Join matchuphistory HS On S.SmallestPointDifferential = HS.pointdiff and (S.projectedwinningmanager = HS.winningmanager or S.projectedwinningmanager = HS.losingmanager) and (S.projectedlosingmanager = HS.losingmanager or s.projectedlosingmanager = HS.winningmanager)
''')

report = pd.read_sql(sql, con=engine.connect())

In [15]:
for index, row in report.iterrows():
    if index == 0: print(f"Week {row['week']} of the {row['season']} season")
    print(f"""
    - {row['projectedwinningmanager']} vs {row['projectedlosingmanager']} 
        {f"- Lifetime record ({row['lifetimewins']}-{row['lifetimelosses']})" if not pd.isna(row['season_s']) else '-First matchup'}
        {f"- Smallest point differential {row['smallestpointdifferential']} ({row['winningmanager_s']} beat {row['losingmanager_s']} {row['winningpoints_s']} to {row['losingpoints_s']} in week {int(row['week_s'])} of the {row['season_s']} season)" if not pd.isna(row['season_s']) else ''}
        {f"- Largest point differential {row['largestpointdifferential']} ({row['winningmanager_l']} beat {row['losingmanager_l']} {row['winningpoints_l']} to {row['losingpoints_l']} in week {int(row['week_l'])} of the {row['season_l']} season)" if not pd.isna(row['season_s']) else ''}""")

Week 11 of the 2025 season

    - Frankie vs Daniel 
        - Lifetime record (4-2)
        - Smallest point differential 20.8 (Frankie beat Daniel 137.0 to 116.2 in week 2 of the 2025 season)
        - Largest point differential 63.98 (Daniel beat Frankie 172.54 to 108.56 in week 1 of the 2024 season)

    - Skyler vs Liam 
        - Lifetime record (0-1)
        - Smallest point differential 20.6 (Liam beat Skyler 126.88 to 106.28 in week 2 of the 2025 season)
        - Largest point differential 20.6 (Liam beat Skyler 126.88 to 106.28 in week 2 of the 2025 season)

    - Nick vs John 
        - Lifetime record (3-3)
        - Smallest point differential 18.9 (Nick beat John 148.46 to 129.56 in week 12 of the 2023 season)
        - Largest point differential 63.64 (John beat Nick 187.34 to 123.7 in week 16 of the 2023 season)

    - Billy vs Joseph 
        - Lifetime record (3-1)
        - Smallest point differential 4.46 (Joseph beat Billy 144.06 to 139.6 in week 9 of the 2024 sea

- Would be cool to see most recent match up and current win streaks.

## SQL

In [ ]:
--SQL: View Matchup History
Select Season, Week, WinningManager, LosingManager, Min(PointDifferential) as PointDiff From (
SELECT to_char(w.week_start, 'YYYY') as Season, w.week, w.is_playoff as Playoffs, w.team_name as WinningTeam, w.manager_nickname as WinningManager, w.points as WinningPoints, l.team_name as LosingTeam, l.manager_nickname as LosingManager, l.points as LosingPoints, 
(ROUND((w.points - l.points)::numeric, 3)) as PointDifferential
FROM public.matchups W
Left Join public.matchups L On W.week = L.Week And W.Week_start = L.Week_Start and W.Week_end = L.Week_end and W.matchup_id = L.matchup_id and W.winner = 'true' and L.Winner = 'false'
)
Where Playoffs = 1
group By Season, Week, WinningManager, LosingManager
Order By PointDiff Asc

In [ ]:
-- Biggest Fleeces
With TradeTotalPoints as(
SELECT T.Season, T.Trade_ID, T.to_team as Team, T.to_team_key as Team_Key, Round(Avg(P.points)::numeric, 4) as Points
FROM public.trades T
Left Join SeasonalPoints P On t.season = p.season and t.player_id = p.player_id
Where T.status = 'successful'
Group By T.Season, T.Trade_ID, T.to_team, T.to_team_key
) 
Select F.Season, F.Trade_ID, F.Team as Team_One, F.Team_Key as Team_One_Key, F.Points as Team_One_Points, S.Team as Team_Two, S.Team_Key as Team_Two_Key, S.Points as Team_Two_Key,
F.Points - S.Points as Differential
From TradeTotalPoints F
Left Join TradeTotalPoints S On F.Season = S.Season and F.Trade_ID = S.Trade_ID and F.Team_Key != S.Team_Key
Where F.Points - S.Points > 0
Group By F.Season, F.Trade_ID, F.Team, F.Team_Key, F.Points, S.Team, S.Team_Key, S.Points
Order By Differential Desc
;

In [ ]:
-- Weekly Report
Select S.*, 
HS.Season as Season_S, HS.Week as Week_S, HS.winningmanager as WinningManager_S, HS.winningpoints as WinningPoints_S,   
HS.losingmanager as LosingManager_S, HS.Losingpoints as LosingPoints_S,
HL.Season as Season_L, HL.Week as Week_L, HL.winningmanager as WinningManager_L, HL.winningpoints as WinningPoints_L,   
HL.losingmanager as LosingManager_L, HL.Losingpoints as LosingPoints_L
From (
Select M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0) as LifetimeWins, COALESCE(R.Losses, RFallBack.Wins, 0) as LifetimeLosses, COALESCE(Min(H.pointdiff), 0) as SmallestPointDifferential, COALESCE(Max(H.pointdiff), 0) as LargestPointDifferential
from vw_currentmatchup M
Left Join lifetimerecord R On M.projectedwinningmanager = R.winningmanager and M.projectedlosingmanager = R.losingmanager
Left Join lifetimerecord RFallBack On M.projectedwinningmanager = RFallBack.losingmanager and M.projectedlosingmanager = RFallBack.winningmanager 
Left Join matchuphistory H On (M.projectedwinningmanager = H.winningmanager or M.projectedwinningmanager = H.losingmanager) and (M.projectedlosingmanager = H.losingmanager or M.projectedlosingmanager = H.winningmanager)
Group By M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0), COALESCE(R.Losses, RFallBack.Wins, 0)
) S
Left Join matchuphistory HL On S.LargestPointDifferential = HL.pointdiff and (S.projectedwinningmanager = HL.winningmanager or S.projectedwinningmanager = HL.losingmanager) and (S.projectedlosingmanager = HL.losingmanager or s.projectedlosingmanager = HL.winningmanager)
Left Join matchuphistory HS On S.SmallestPointDifferential = HS.pointdiff and (S.projectedwinningmanager = HS.winningmanager or S.projectedwinningmanager = HS.losingmanager) and (S.projectedlosingmanager = HS.losingmanager or s.projectedlosingmanager = HS.winningmanager)
;

In [ ]:
-- Adding Most recent match up
Select S.*, 
HS.Season as Season_S, HS.Week as Week_S, HS.winningmanager as WinningManager_S, HS.winningpoints as WinningPoints_S,   
HS.losingmanager as LosingManager_S, HS.Losingpoints as LosingPoints_S,
HL.Season as Season_L, HL.Week as Week_L, HL.winningmanager as WinningManager_L, HL.winningpoints as WinningPoints_L,   
HL.losingmanager as LosingManager_L, HL.Losingpoints as LosingPoints_L,
HLAST.Season as Season_Last,
HLAST.Week as Week_Last,
HLAST.winningmanager as WinningManager_Last,
HLAST.winningpoints as WinningPoints_Last,
HLAST.losingmanager as LosingManager_Last,
HLAST.losingpoints as LosingPoints_Last
From (
Select M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0) as LifetimeWins, COALESCE(R.Losses, RFallBack.Wins, 0) as LifetimeLosses, COALESCE(Min(H.pointdiff), 0) as SmallestPointDifferential, COALESCE(Max(H.pointdiff), 0) as LargestPointDifferential
from vw_currentmatchup M
Left Join lifetimerecord R On M.projectedwinningmanager = R.winningmanager and M.projectedlosingmanager = R.losingmanager
Left Join lifetimerecord RFallBack On M.projectedwinningmanager = RFallBack.losingmanager and M.projectedlosingmanager = RFallBack.winningmanager 
Left Join matchuphistory H On (M.projectedwinningmanager = H.winningmanager or M.projectedwinningmanager = H.losingmanager) and (M.projectedlosingmanager = H.losingmanager or M.projectedlosingmanager = H.winningmanager)
Group By M.Season, M.Week, M.projectedwinningmanager, M.projectedlosingmanager, COALESCE(R.Wins, RFallBack.Losses, 0), COALESCE(R.Losses, RFallBack.Wins, 0)
) S
Left Join matchuphistory HL On S.LargestPointDifferential = HL.pointdiff and (S.projectedwinningmanager = HL.winningmanager or S.projectedwinningmanager = HL.losingmanager) and (S.projectedlosingmanager = HL.losingmanager or s.projectedlosingmanager = HL.winningmanager)
Left Join matchuphistory HS On S.SmallestPointDifferential = HS.pointdiff and (S.projectedwinningmanager = HS.winningmanager or S.projectedwinningmanager = HS.losingmanager) and (S.projectedlosingmanager = HS.losingmanager or s.projectedlosingmanager = HS.winningmanager)
Left Join matchuphistory HLAST On 
    (S.projectedwinningmanager = HLAST.winningmanager or S.projectedwinningmanager = HLAST.losingmanager)
    and (S.projectedlosingmanager = HLAST.losingmanager or S.projectedlosingmanager = HLAST.winningmanager)
    and HLAST.Week = (
        Select Max(H2.Week)
        From matchuphistory H2
        Where 
            (S.projectedwinningmanager = H2.winningmanager or S.projectedwinningmanager = H2.losingmanager)
            and (S.projectedlosingmanager = H2.losingmanager or S.projectedlosingmanager = H2.winningmanager)
            and H2.Season = (
                Select Max(H3.Season)
                From matchuphistory H3
                Where 
                    (S.projectedwinningmanager = H3.winningmanager or S.projectedwinningmanager = H3.losingmanager)
                    and (S.projectedlosingmanager = H3.losingmanager or S.projectedlosingmanager = H3.winningmanager)
            )
    )